# Project 3.2 — Projectile Motion with Drag
**Topic:** RK4 integration of 2-D projectile motion with velocity-dependent drag forces. The role of drag regime, optimal angle, and terminal velocity.


---
## 1. Set Up

### Physical Motivation
The equation of motion for a projectile with drag:

$$m\ddot{\mathbf{r}} = -mg\hat{j} - b|\mathbf{v}|^{n-1}\mathbf{v}$$

Two physically distinct regimes, characterised by the **Reynolds number** $Re = \rho v L/\mu$:

| Regime | $n$ | Force law | Physical context |
|--------|-----|-----------|-----------------|
| **Stokes (viscous)** | $n=1$ | $F_d = -b\mathbf{v}$ | $Re \ll 1$: fog droplets, bacteria, slow settling |
| **Newtonian (inertial)** | $n=2$ | $F_d = -b|\mathbf{v}|\mathbf{v}$ | $Re \gg 1$: baseballs, artillery, aircraft |

The parameter `n_drag` interpolates between these. **Critical physical difference**: terminal velocity.
- Linear ($n=1$): $v_t = mg/b$ — approached exponentially.
- Quadratic ($n=2$): $v_t = \sqrt{mg/b}$ — faster growth of drag, lower terminal velocity for same $b$.

### No Analytic Solution for $n=2$
For linear drag, the trajectory has an analytic solution. For quadratic drag, there is **no closed form** — numerical integration is required. This is why computational methods are indispensable for realistic ballistics.

### ⚠️ Key Subtleties
1. **Drag acts opposite to velocity** (not just vertical): $\mathbf{F}_d \propto -\hat{v}$, so both $a_x$ and $a_y$ are affected.
2. **Optimal angle is not 45°** with drag. The optimal angle is less than 45° and decreases as drag increases.
3. **Ground detection** uses linear interpolation — the actual landing point is between the last two RK4 steps.


In [ ]:
import os, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

OUT = Path("output_projectile_drag")
OUT.mkdir(exist_ok=True)
np.set_printoptions(precision=6, suppress=True)
print(f"Output directory: {OUT.resolve()}")


---
## 2. Functions

### `projectile_rhs(t, y, p)`
State vector: $y = [x, y_{\text{pos}}, v_x, v_y]$ — position and velocity in 2-D.

```python
speed = np.hypot(vx, vy)
drag_factor = b * speed**(n_drag - 1.0) / m
ax_drag = -drag_factor * vx
ay_drag = -drag_factor * vy
```
**Physics**: the drag acceleration magnitude is $a_d = b|v|^{n-1}/m$ and its direction is $-\hat{v} = -(v_x, v_y)/|v|$. So $a_{x,\text{drag}} = -a_d \cdot v_x/|v| \cdot |v| = -bv_x|v|^{n-1}/m$. This correctly decomposes the drag along each coordinate.

`np.hypot(vx, vy)` computes $\sqrt{v_x^2+v_y^2}$ with better numerical stability than `np.sqrt(vx**2+vy**2)` (avoids overflow for very large velocities).

```python
return np.array([vx, vy, ax_drag, -g + ay_drag], dtype=float)
```
The $-g$ only appears in the $y$-acceleration (gravity is vertical). The $x$-direction has **no gravity** — only drag. This is why projectile range with drag is always less than without: drag reduces $v_x$ throughout the flight.

### Ground Detection — Linear Interpolation
```python
frac = -y1 / (y2 - y1)
range_est = x1 + frac * (x2 - x1)
```
When $y$ crosses zero between steps (y1 > 0, y2 < 0), the landing point is interpolated linearly. **This introduces an error of $\mathcal{O}(\Delta t^2)$** — quadratic in the step size. For $\Delta t = 0.005\,\text{s}$ and $v_y \approx 10\,\text{m/s}$, the position error is $\sim (v_y \Delta t)^2/(2g) \approx 3\times 10^{-4}\,\text{m}$ — negligible for most purposes.

### 🔧 Improvement Directions
1. **Optimal launch angle**: sweep $\theta \in [1°, 89°]$ and plot range vs $\theta$. For drag, the optimum is $< 45°$, and finding it numerically is a good optimisation exercise.
2. **Variable air density**: $\rho(h) = \rho_0 e^{-h/H}$ where $H \approx 8.5\,\text{km}$. The drag coefficient becomes altitude-dependent.
3. **Magnus effect** (spinning ball): add a lift force $\mathbf{F}_{\text{Magnus}} = k_M \boldsymbol\omega \times \mathbf{v}$ perpendicular to velocity.
4. **Bisection landing refinement**: once the ground crossing is detected, bisect the interval to find landing to machine precision.

### ⚠️ Known Weaknesses
- **Point particle**: ignores the projectile's shape, rotation, and cross-sectional area.
- **Constant $g$**: Earth's gravity decreases as $1/r^2$; for high-altitude trajectories, this matters.
- **2-D only**: wind, Coriolis force, and lateral drift require 3-D.


In [ ]:
def projectile_rhs(t, y, p):
    x_pos, y_pos, vx, vy = y
    g      = p.get("g", 9.81)
    m      = p.get("mass", 1.0)
    b      = p.get("b", 0.0)
    n_drag = p.get("n_drag", 1.0)

    speed  = np.hypot(vx, vy)
    if speed == 0:
        ax_drag, ay_drag = 0.0, 0.0
    else:
        df = b * speed**(n_drag - 1.0) / m
        ax_drag = -df * vx
        ay_drag = -df * vy

    return np.array([vx, vy, ax_drag, -g + ay_drag], dtype=float)

def rk4_step(f, t, y, dt, params):
    k1 = f(t,           y,             params)
    k2 = f(t + 0.5*dt,  y + 0.5*dt*k1, params)
    k3 = f(t + 0.5*dt,  y + 0.5*dt*k2, params)
    k4 = f(t + dt,       y + dt*k3,    params)
    return y + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)

def integrate_projectile(v0, theta_deg, params, dt=0.005, t_max=20.0):
    theta = np.deg2rad(theta_deg)
    y0    = np.array([0., 0., v0*np.cos(theta), v0*np.sin(theta)])

    t_vals, states = [0.0], [y0]
    t, y = 0.0, y0.copy()

    while t < t_max:
        y_next = rk4_step(projectile_rhs, t, y, dt, params)
        t += dt
        states.append(y_next)
        t_vals.append(t)
        if y_next[1] < 0 and t > 0:
            break
        y = y_next

    arr   = np.array(states)
    t_arr = np.array(t_vals)

    # Linear interpolation for landing point
    if arr[-1, 1] < 0 and len(arr) >= 2:
        y1, y2 = arr[-2, 1], arr[-1, 1]
        x1, x2 = arr[-2, 0], arr[-1, 0]
        frac    = -y1 / (y2 - y1)
        range_est = x1 + frac*(x2 - x1)
    else:
        range_est = arr[-1, 0]

    return t_arr, arr, range_est


---
## 3 & 4. Calculation & Writeouts

### What to Watch
- **No drag**: trajectory is a perfect parabola. Range $= v_0^2\sin(2\theta)/g$. Maximum at $\theta = 45°$.
- **Linear drag**: trajectory is asymmetric (faster ascent than descent) and range is reduced.
- **Quadratic drag**: much stronger effect at high speeds. Terminal velocity limits descent speed.
- **Optimal angle with drag**: find by scanning $\theta$ values. Expect $\theta_{\text{opt}} < 45°$.


In [ ]:
v0     = 50.0     # launch speed (m/s)
angles = [15, 30, 45, 60, 75]
params_list = [
    ("No drag",       {"g": 9.81, "b": 0.0}),
    ("Linear drag",   {"g": 9.81, "b": 0.05, "n_drag": 1.0}),
    ("Quadratic drag",{"g": 9.81, "b": 0.005,"n_drag": 2.0}),
]
colors = ['steelblue','tomato','darkgreen']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (label, params), col in zip(axes, params_list, colors):
    for theta in angles:
        t_arr, arr, rng_est = integrate_projectile(v0, theta, params)
        mask = arr[:, 1] >= 0
        ax.plot(arr[mask, 0], arr[mask, 1], color=col,
                lw=1.5, alpha=0.6 + 0.06*theta/15,
                label=f"θ={theta}°" if label=="No drag" else None)
    ax.set_xlabel("Range (m)")
    ax.set_ylabel("Height (m)")
    ax.set_title(label)
    ax.set_ylim(bottom=0)

# Optimal angle search (quadratic drag)
params_quad = {"g": 9.81, "b": 0.005, "n_drag": 2.0}
thetas      = np.arange(5, 85, 1)
ranges_quad = [integrate_projectile(v0, th, params_quad)[2] for th in thetas]
ranges_none = [integrate_projectile(v0, th, {"g":9.81,"b":0.0})[2] for th in thetas]
opt_quad    = thetas[np.argmax(ranges_quad)]
print(f"Optimal angle (no drag):        45.0°  (theory)")
print(f"Optimal angle (quadratic drag): {opt_quad:.1f}°")
print(f"Max range (no drag):    {max(ranges_none):.2f} m  (theory: {v0**2/9.81:.2f} m)")
print(f"Max range (quad. drag): {max(ranges_quad):.2f} m")

axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT / "projectile_drag.png", dpi=150, bbox_inches='tight')
plt.show()


---
## 6. Sanity Checks

| Test | Formula | Note |
|------|---------|------|
| No-drag range at 45° | $v_0^2/g$ | Analytic result |
| Monotone $y\geq0$ until landing | — | No sub-ground trajectory |
| Landing $y \approx 0$ | $|y_{\text{final}}| < g\Delta t^2$ | Ground detection accuracy |
| Parabolic shape (no drag) | Fit $y = ax + bx^2$, check $R^2 > 0.9999$ | Trajectory is parabola |


In [ ]:
print("=" * 55)
print("SANITY CHECKS — Project 03.2 Projectile Drag")
print("=" * 55)

# 1. No-drag range at 45°
t_arr, arr, rng45 = integrate_projectile(50.0, 45.0, {"g":9.81,"b":0.0})
rng_theory = 50.0**2 / 9.81
ok = abs(rng45 - rng_theory) / rng_theory < 0.001
print(f"\n1. Range at 45° (no drag): {rng45:.3f} m, theory: {rng_theory:.3f} m  "
      f"err={abs(rng45-rng_theory)/rng_theory*100:.3f}%  {'✓' if ok else '✗'}")

# 2. All y-values non-negative until last step
mask = arr[:, 1] < -0.5
ok2 = not np.any(mask[:-2])   # allow last 2 steps to go slightly negative
print(f"2. y ≥ 0 throughout (except landing step): {'✓' if ok2 else '✗'}")

# 3. Final y ≈ 0
ok3 = abs(arr[-1, 1]) < 0.5   # within half a meter
print(f"3. Final y = {arr[-1,1]:.4f} m  {'✓' if ok3 else '✗'}")

# 4. Parabola check (no drag, 45°)
x_traj = arr[:-1, 0][arr[:-1,1] >= 0]
y_traj = arr[:-1, 1][arr[:-1,1] >= 0]
if len(x_traj) > 5:
    p_fit = np.polyfit(x_traj, y_traj, 2)
    y_fit = np.polyval(p_fit, x_traj)
    ss_res = np.sum((y_traj - y_fit)**2)
    ss_tot = np.sum((y_traj - y_traj.mean())**2)
    R2 = 1 - ss_res/ss_tot
    ok4 = R2 > 0.9999
    print(f"4. Parabola fit R² = {R2:.8f}  {'✓' if ok4 else '✗'}")

print("\n" + "=" * 55)
